In [4]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch import optim
from torch.utils.data import Subset
from model import DayCentModel
from data import DayCentDataset
import os
import wandb
from utils import evaluate

In [5]:
# Reproducibility
RND_SEED = 42
np.random.seed(RND_SEED)
torch.manual_seed(RND_SEED)

In [6]:
EXPERIMENT_ID = "experiment11"
INPUT_NPY = f"/users/6/mehta423/daycent/data/{EXPERIMENT_ID}/train_X.npy"
OUTPUT_NPY = f"/users/6/mehta423/daycent/data/{EXPERIMENT_ID}/train_Y.npy"
INIT_COND = "/users/6/mehta423/daycent/data/SAS_KGML_090925/InputData/initial_site_conditions.xlsx"
OUTPUT_DIR = f"/users/6/mehta423/daycent/output/{EXPERIMENT_ID}"

In [22]:
# ----------------------
# Config
# ----------------------
BATCH_SIZE = 2048
EPOCHS = 100
LR = 1e-2
DEVICE = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    # Set the wandb project where this run will be logged.
    project="daycent",
    name="experiment11/quadrant-subset-30/100-scenarios",
    notes="For this the holdout is quadrant wise. A subset for Q1 is for training and Q2, Q3, Q4 are for testing. Using 30 out of 100 preprocessed scenarios.",
    config={
        "learning_rate": LR,
        "architecture": "LSTM with Attention",
        "dataset": "20 Scenarios, 20 Points",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE
    },
)

In [8]:
# ----------------------
# Dataloader
# ----------------------
# check dataset
dataset = DayCentDataset(INPUT_NPY, OUTPUT_NPY, INIT_COND, apply_scaling=True)

0


In [9]:
len(dataset)

50000

In [17]:
NUM_SCENARIOS = 100
UNIT_SIZE = int(len(dataset) / NUM_SCENARIOS)
train_size = int(UNIT_SIZE * (NUM_SCENARIOS * 0.3))
val_size = int(UNIT_SIZE * (NUM_SCENARIOS * 0.2))
test_size = int(UNIT_SIZE * (NUM_SCENARIOS * 0.1))

# 2) Create index arrays for each split
train_idx = np.arange(0, train_size)
val_idx = np.arange(train_size, train_size + val_size)
test_idx = np.arange(train_size + val_size, train_size + val_size + test_size)

# 3) Wrap subsets
train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
test_ds  = Subset(dataset, test_idx)

# 4) Create loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Dataset sizes — total: {len(dataset)}, train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

Dataset sizes — total: 50000, train: 15000, val: 10000, test: 5000


In [18]:
# infer input dim
sample = dataset[0]
seq_feat_dim = sample["sequence"].shape[1]  # #features
init_dim = sample["init_cond"].shape[0]
year_dim = sample["year_enc"].shape[0]

print(f"Input feature dim: {seq_feat_dim}, init cond dim: {init_dim}, year enc dim: {year_dim}")

Input feature dim: 20, init cond dim: 245, year enc dim: 16


In [23]:

model = DayCentModel(input_dim=seq_feat_dim, init_dim=init_dim, year_dim=year_dim)
model.to(DEVICE)

DayCentModel(
  (init_proj): Linear(in_features=261, out_features=32, bias=True)
  (daily_proj): Linear(in_features=20, out_features=32, bias=True)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True)
  (somsc_attn): AttentionPooling(
    (attn): Linear(in_features=128, out_features=1, bias=True)
    (proj): Linear(in_features=128, out_features=128, bias=True)
  )
  (yield_attn): AttentionPooling(
    (attn): Linear(in_features=128, out_features=1, bias=True)
    (proj): Linear(in_features=128, out_features=128, bias=True)
  )
  (somsc_head): Linear(in_features=128, out_features=1, bias=True)
  (yield_head): Linear(in_features=128, out_features=1, bias=True)
)

In [24]:
# ----------------------
# Optimizer & scheduler
# ----------------------
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)

In [25]:
# ----------------------
# Training loop with loss tracking
# ----------------------
best_val_loss = float('inf')


# Initialize loss tracking lists
train_losses = []
val_somsc_losses = []
val_yield_losses = []
val_total_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        # move to device
        batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        optimizer.zero_grad()
        out = model(batch)

        # --- SOMSC loss ---
        somsc_target = batch["somsc"]              # (B,12)
        somsc_mask = batch["somsc_mask"]           # (B,12)

        # compute masked MSE
        somsc_loss = ((out["somsc_pred"].squeeze(-1) - somsc_target)**2 * somsc_mask).sum() / somsc_mask.sum()

        # --- Yield loss ---
        yield_target = batch["yield"]              # (B,)
        yield_mask = batch["yield_mask"]           # (B,)
        yield_loss = ((out["yield_pred"] - yield_target)**2 * yield_mask).sum() / yield_mask.sum()

        # --- total loss ---
        alpha = 1.0  # weight for SOMSC loss
        beta = 1.0   # weight for Yield loss
        loss = alpha*somsc_loss + beta*yield_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch["sequence"].size(0)

    total_loss /= len(dataset)
    train_losses.append(total_loss)
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss:.4f}")
    somsc_loss_val, yield_loss_val = evaluate(model, val_loader, DEVICE)
    print(f"  Val SOMSC Loss: {somsc_loss_val:.4f}, Yield Loss: {yield_loss_val:.4f}")

    if yield_loss_val < best_val_loss:
        best_val_loss = yield_loss_val
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pth"))
        print(f"Saved best model at epoch {epoch} with val_loss: {yield_loss_val:.4f}")
    
    # Track validation losses
    val_somsc_losses.append(somsc_loss_val)
    val_yield_losses.append(yield_loss_val)
    val_total_losses.append(somsc_loss_val + yield_loss_val)

    run.log({
        "epoch": epoch + 1,
        "train_loss": total_loss,
        "val_somsc_loss": somsc_loss_val,
        "val_yield_loss": yield_loss_val,
        "val_total_loss": somsc_loss_val + yield_loss_val,
        "learning_rate": optimizer.param_groups[0]['lr'],
    })

    scheduler.step(total_loss)


Epoch 1/100 - Loss: 0.6855
  Val SOMSC Loss: 0.2857, Yield Loss: 1.0675
Saved best model at epoch 0 with val_loss: 1.0675
Epoch 2/100 - Loss: 0.3927
  Val SOMSC Loss: 0.2210, Yield Loss: 1.0541
Saved best model at epoch 1 with val_loss: 1.0541
Epoch 3/100 - Loss: 0.3623
  Val SOMSC Loss: 0.1971, Yield Loss: 0.9864
Saved best model at epoch 2 with val_loss: 0.9864
Epoch 4/100 - Loss: 0.3356
  Val SOMSC Loss: 0.1913, Yield Loss: 0.8564
Saved best model at epoch 3 with val_loss: 0.8564
Epoch 5/100 - Loss: 0.3173
  Val SOMSC Loss: 0.1931, Yield Loss: 0.8407
Saved best model at epoch 4 with val_loss: 0.8407
Epoch 6/100 - Loss: 0.3059
  Val SOMSC Loss: 0.1691, Yield Loss: 0.7628
Saved best model at epoch 5 with val_loss: 0.7628
Epoch 7/100 - Loss: 0.3172
  Val SOMSC Loss: 0.1779, Yield Loss: 0.9574
Epoch 8/100 - Loss: 0.3274
  Val SOMSC Loss: 0.1565, Yield Loss: 0.8860
Epoch 9/100 - Loss: 0.3041
  Val SOMSC Loss: 0.1593, Yield Loss: 0.7618
Saved best model at epoch 8 with val_loss: 0.7618
Ep

In [28]:
run.finish()
